# HSV-2 research showcase

This notebook presents the standardized ViralSafeTarget case study. It deliberately keeps sequence targetability, source-linked essentiality, predicted protein disruption, and evidence coverage separate. The outputs are computational research hypotheses, not treatment recommendations or wet-lab protocols.

## Reproduce the showcase

From the repository root, run:

```bash
vst profiles validate --virus-profile configs/viruses/hsv2.yaml --host-profile configs/hosts/human_grch38.yaml --nuclease-profile configs/nucleases/spcas9.yaml
vst showcase build --virus-profile configs/viruses/hsv2.yaml --host-profile configs/hosts/human_grch38.yaml --nuclease-profile configs/nucleases/spcas9.yaml --out-dir reports/hsv2_showcase
```

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

def show_image(path):
    image = plt.imread(path)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.imshow(image)
    ax.axis('off')
    plt.show()

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
OUT = ROOT / 'reports' / 'hsv2_showcase'
required = [
    OUT / 'candidates_evidence_aware.csv',
    OUT / 'deep_screening_panel.csv',
    OUT / 'strategy_comparison.csv',
    OUT / 'research_findings.csv',
    OUT / 'run_manifest.json',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Build the showcase first. Missing: ' + ', '.join(missing))

## Potentially novel computational findings

This table turns the main observations into auditable research hypotheses. Each row keeps its computational support beside the limitation that prevents it from being presented as biological validation or a literature-wide novelty claim.

In [ ]:
research_findings = pd.read_csv(OUT / 'research_findings.csv')
display(research_findings)

## Provenance and data funnel

The manifest records profile sources, checksums, parameters, and output counts. The funnel is the presentation-level summary of the completed HSV-2 case study.

In [ ]:
manifest = json.loads((OUT / 'run_manifest.json').read_text(encoding='utf-8'))
display(manifest['profile_summary'])
show_image(OUT / 'figures' / 'pipeline_funnel.png')

## Multi-objective gene landscape

A gene can be easy to target yet weakly supported biologically. This view prevents those concepts from being collapsed into a single therapeutic-looking score.

In [ ]:
candidates = pd.read_csv(OUT / 'candidates_evidence_aware.csv')
show_image(OUT / 'figures' / 'gene_landscape.png')
display(
    candidates[[
        'gene_name', 'candidate_id', 'post_human_rank', 'pareto_front',
        'candidate_predicted_disruption_score', 'evidence_tier'
    ]].head(15)
)

## Balanced deep panel

The deep panel applies the same per-gene quota after non-dominated sorting. It is an auditable shortlist for further expert review, not a set of recommended interventions.

In [ ]:
deep_panel = pd.read_csv(OUT / 'deep_screening_panel.csv')
show_image(OUT / 'figures' / 'deep_panel_heatmap.png')
display(deep_panel.groupby(['primary_category', 'gene_name']).size().rename('candidate_count'))

## Comparison sets

The comparison sets expose the trade-offs between ranking-only, disruption-focused, evidence-anchored, and mechanistically diverse selection. They do not model joint editing, delivery, safety, or efficacy.

In [ ]:
strategies = pd.read_csv(OUT / 'strategy_comparison.csv')
show_image(OUT / 'figures' / 'strategy_comparison.png')
display(strategies)

## Interpretation boundary

The defensible finding is methodological: virus-wide sequence ranking and biological target evidence answer different questions, and a transparent multi-objective workflow makes that distinction visible. Read `FINDINGS.md`, `METHODS.md`, and `LIMITATIONS.md` before presenting any candidate-level result.